# Open Powerlifting Data Preprocessing
Name: Collin Joseph<br>

### Dataset: OpenPowerlifting Powerlifting Database
Retrieved from: [powerlifting-database](https://www.kaggle.com/datasets/dansbecker/powerlifting-database)<br>
This dataset is comprises performance stats and characteristics of powerlifting athletes from various powerlifting meets.<br>
There are two files in this version of the dataset: `openpowerlifting.csv` which contains athlete characteristics and peformance information for 3 events: benchpress, squats and deadlift, and `meets.csv` which contains information about the powerlifting meets such as organization and location. I will only be focusing on the characteristics of the athletes and will therefore only be using `openpowerlifting.csv`<br>

### Objective
I will treat `TotalKg` as the target variable for this regression task. <br>
`TotalKg` is the sum of each athletes best performances in each event. This makes it a useful overall measure of the athlete's performance. <br>
Characteristics such as `Sex`, `Age`, `BodyweightKg` and `Equipment` could have an impact on the athlete's performance so they will be analysed and used to generate or as features for my regression models.

In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df_opl = pd.read_csv("openpowerlifting.csv")

### Data Preprocessing

#### Columns of Interest
Drop Name and MeetID since the will not be used in analysis.

In [3]:
df_opl = df_opl.drop(columns=["Name", "MeetID"])

Remove rows missing best performance scores. We only want samples where the athlete competed in all 3 events since `TotalKg` is the sum of the athlete's best performances. <br>
Afterwards drop the individual event columns. Using them as predictors of `TotalKg` would be pointless due to the nature of the target variable.

In [4]:
df_opl = df_opl.dropna(subset=["BestBenchKg", "BestSquatKg", "BestDeadliftKg"])
df_opl = df_opl.drop(columns=["BestBenchKg", "BestSquatKg", "BestDeadliftKg"])

Drop other performance related columns.

In [5]:
df_opl = df_opl.drop(columns=["Bench4Kg", "Squat4Kg", "Deadlift4Kg"])

In [6]:
df_opl = df_opl.drop(columns=["Place", "Wilks"])

#### Missing Data
In this section, the proportion of missing data will be evaluated and missing data will be addressed based on what data is available.

In [7]:
df_opl.isnull().sum() / df_opl.shape[0]

Sex              0.000000
Equipment        0.000000
Age              0.624798
Division         0.047411
BodyweightKg     0.002084
WeightClassKg    0.006292
TotalKg          0.005725
dtype: float64

##### Performance Data: TotalKg
Drop records without a `TotalKg` score since we cannot use them to build our regression model.

In [8]:
df_opl = df_opl.dropna(subset = ["TotalKg"])

##### Bodyweight
Fill the 0.6% missing values with average weight grouped by sex.

In [9]:
df_opl["BodyweightKg"] = df_opl.groupby("Sex")["BodyweightKg"].transform(lambda x: x.fillna(x.mean()))

##### Weight Class
Drop weight class since bodyweight provides more specific information and makes it redundant.

In [10]:
df_opl = df_opl.drop(columns=["WeightClassKg"])

##### Age
Over 61% of age entries are missing. I will attempt to infer age values from division since many have specific age ranges. <br>

In [11]:
# Determine division age median using regex where possible.
@np.vectorize
def median_age_from_division(s):
    pattern = re.compile(r".*(\b\d{2}\b)-(\b\d{2}\b).*")
    m = pattern.match(str(s))
    if m:
        median_age = (int(m.groups()[0]) + int(m.groups()[1])) / 2
    else:
        median_age = np.nan
    return median_age

division_median = df_opl["Division"].transform(median_age_from_division)

In [12]:
# Replace missing age values with division median where possible
df_opl["Age"] = df_opl["Age"].fillna(division_median)

In [13]:
percAgeMissing = 100 * df_opl["Age"].isnull().sum() / df_opl.shape[0]
print(f"{percAgeMissing} % of age values missing")

55.67167725389072 % of age values missing


Drop records with Age still missing.

In [14]:
df_opl = df_opl.dropna(subset=["Age"])

Drop Division column now since we no longer have a use for it.

In [15]:
df_opl = df_opl.drop(columns=["Division"])

Confirm no more missing values.

In [16]:
df_opl.isnull().sum() / df_opl.shape[0]

Sex             0.0
Equipment       0.0
Age             0.0
BodyweightKg    0.0
TotalKg         0.0
dtype: float64

In [17]:
df_opl.shape[0]

126865

#### Data Types
Investigate and correct data types

In [18]:
df_opl.dtypes

Sex              object
Equipment        object
Age             float64
BodyweightKg    float64
TotalKg         float64
dtype: object

##### Categories 
Convert sex and equipment to a categorical numerical variables

In [19]:
# Change Sex category to binary indicator since dataset only has Male and Female entries.
df_opl["Sex"] = df_opl["Sex"].astype("category")
df_opl = pd.get_dummies(df_opl, prefix="Sex", columns=["Sex"], dtype="float", drop_first=True)

In [20]:
# Convert Equipment category to numerical indicators, we drop the first value since they are mutually exclusive.
df_opl["Equipment"] = df_opl["Equipment"].astype("category")
df_opl = pd.get_dummies(df_opl, columns=["Equipment"], prefix="Equipment", dtype="float", drop_first=True)

In [21]:
df_opl.rename(columns={"Equipment_Single-ply":"Equipment_SinglePly"}, inplace=True)

In [22]:
df_opl.columns

Index(['Age', 'BodyweightKg', 'TotalKg', 'Sex_M', 'Equipment_Raw',
       'Equipment_SinglePly', 'Equipment_Wraps'],
      dtype='object')

#### Normalization
Normalize `Age` and `BodyweightKg` to \[0, 1\] range

In [23]:
from sklearn.preprocessing import MinMaxScaler

In [24]:
opl_mms = MinMaxScaler()
df_opl[["Age", "BodyweightKg"]] = opl_mms.fit_transform(df_opl[["Age", "BodyweightKg"]])

In [25]:
df_opl.head()

,Age,BodyweightKg,TotalKg,Sex_M,Equipment_Raw,Equipment_SinglePly,Equipment_Wraps
0,0.465116,0.162620,138.35,0.0,0.0,0.0,1.0
1,0.406977,0.157627,401.42,0.0,0.0,1.0,0.0
2,0.406977,0.157627,401.42,0.0,0.0,1.0,0.0
5,0.244186,0.175492,392.36,0.0,0.0,0.0,1.0
6,0.616279,0.197939,383.28,0.0,1.0,0.0,0.0


#### Target Variable
Observe statistics of target variable `TotalKg` in dataset.

In [26]:
df_opl["TotalKg"].describe()

count    126865.000000
mean        495.871806
std         172.904138
min          38.600000
25%         352.500000
50%         497.500000
75%         614.980000
max        1363.050000
Name: TotalKg, dtype: float64

#### Save Cleaned and Processed Dataset

In [27]:
df_opl.to_csv("opl_clean.csv", index=False)